# ♟️ Chess Numba — Google Colab LMR Phase 1 (Tail 0.1 專用獨立驗證執行器)

本 Notebook 專門用於在**獨立的 Google Colab Kernel / 虛擬機**上運行 **Polyak-Ruppert Tail 0.1**（取最後 10% 輪數 / 1,200 局平均）的最佳候選參數之 **L2 深度對抗驗證**（300,000 nodes, 600 局, `data/openings.epd`）。

### 🎯 驗證目標
* **候選參數**：`tail 0.1` 候選組（`435 / 205 / 1848 / 1188 / 35 / 113`）
* **對照基準**：原始 Baseline（`463 / 167 / 2117 / 1035 / 38 / 100`）
* **測試規格**：600 局 @ 300,000 nodes，SPRT(0.0, 3.0) 提早決策
* **輸出檔案**：`tournament_analysis/l2_verify_lmr_p1_deep_tail01.json`（與 tail 0.5 及 tail 0.3 完全隔離不衝突）

In [1]:
# 1. 檢視 Colab 雲端硬體規格
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!python3 --version

CPU(s):                                  8
On-line CPU(s) list:                     0-7
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0-7
Python 3.13.16


In [2]:
# 2. 克隆/拉取最新代碼庫並安裝相依套件
import os
branch = "feat-initial-bitboard-setup"

if not os.path.exists("/content/chess_numba"):
    %cd /content
    !git clone -b {branch} https://github.com/b08203031/chess_numba.git

%cd /content/chess_numba
!git checkout -f {branch}
!rm -f tournament_analysis/l2_verify_lmr_p1_deep.json tune_search/candidate_lmr_p1_deep.json
!git pull origin {branch}
!pip install -q numba chess numpy

/content
Cloning into 'chess_numba'...
remote: Enumerating objects: 3391, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 3391 (delta 97), reused 105 (delta 88), pack-reused 3262 (from 1)
Receiving objects: 100% (3391/3391), 175.69 MiB | 38.82 MiB/s, done.
Resolving deltas: 100% (2321/2321), done.
/content/chess_numba
Already on 'feat-initial-bitboard-setup'
Your branch is up to date with 'origin/feat-initial-bitboard-setup'.
From https://github.com/b08203031/chess_numba
 * branch            feat-initial-bitboard-setup -> FETCH_HEAD
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 53.9 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done


In [3]:
# 3. 測試環境與自動準備 80 輪 SPSA 日誌（全新虛擬機 1 秒還原）
import os
import sys
import json

%cd /content/chess_numba
sys.path.insert(0, "/content/chess_numba")

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"偵測到 Colab CPU 核心數: {cpu_count}，並行度設為: {concurrency}")

log_file = "tune_search/logs/lmr_p1_deep_l1.jsonl"
state_file = "tune_search/states/lmr_p1_deep_l1.json"

# 若為全新 VM，自動從內建壓縮快照還原完整的 80 輪 (12,000 局) 歷史日誌！
if not os.path.exists(log_file) or not os.path.exists(state_file):
    import tools.recover_lmr_p1_iter28 as recover
    recover.restore_checkpoint(concurrency=concurrency, force=True)
else:
    print(f"✅ 偵測到已有日誌檔：{log_file}")

from tune_search.search_param_registry import get_default_params
target_params = [
    "LMR_BASE_OFFSET",
    "LMR_HISTORY_SCALE",
    "LMR_CUTNODE_BONUS",
    "LMR_TTCAPTURE_BONUS",
    "LMR_MOVECOUNT_FACTOR",
    "LMR_TABLE_SCALE_PERCENT"
]
print("✅ 基準參數預設值（Baseline）：")
print(get_default_params(names=target_params))
# 4. 分析 SPSA 調參成果，以 Polyak-Ruppert 末端 10% 平均（tail 0.1，後 8 輪 / 1,200 局）產出候選參數
!python tune_search/summarize_run.py tune_search/logs/lmr_p1_deep_l1.jsonl --tail 0.1 --out tune_search/candidate_lmr_p1_deep_tail01.json
print("\n=== 候選參數 JSON 內容 (tail 0.1) ===")
!cat tune_search/candidate_lmr_p1_deep_tail01.json

/content/chess_numba
偵測到 Colab CPU 核心數: 8，並行度設為: 7
[recover] Restoring full Iter 1-80 history (12,000 games)...
[recover] Successfully restored all 80 iterations to tune_search/logs/lmr_p1_deep_l1.jsonl.
[recover] Successfully created checkpoint at tune_search/states/lmr_p1_deep_l1.json (iteration=80, concurrency=7).
✅ 基準參數預設值（Baseline）：
{'LMR_BASE_OFFSET': 463, 'LMR_HISTORY_SCALE': 167, 'LMR_CUTNODE_BONUS': 2117, 'LMR_TTCAPTURE_BONUS': 1035, 'LMR_MOVECOUNT_FACTOR': 38, 'LMR_TABLE_SCALE_PERCENT': 100}
iterations=80 games=12000 mean_score=0.4990
score sd observed=0.0302 noise-only expected=0.0327 ratio=0.92
  -> ratio ~1: batch scores are indistinguishable from sampling noise; this run carries (almost) no gradient signal. Use more games per batch / more batches before trusting any parameter movement.

param                             default      last  tail_avg moved%rng  drift_z
LMR_BASE_OFFSET                       463     434.9     435.5     -6.24    -1.05  (noise)
LMR_HISTORY_SCALE

In [4]:
# 5. 啟動 L2 深度 SPRT 實戰驗證（300,000 nodes, 600 局，主開局庫獨立驗證）
# 對照基準明確指定為原始 Baseline (tune_search/baseline_lmr_p1.json)
!python tools/run_search_param_match.py \
  --params-json tune_search/candidate_lmr_p1_deep_tail01.json \
  --baseline-json tune_search/baseline_lmr_p1.json \
  --nodes 300000 \
  --games 600 \
  --openings data/openings.epd \
  --concurrency {concurrency} \
  --sprt 0,3 \
  --batch-games 50 \
  --adjudicate \
  --out tournament_analysis/l2_verify_lmr_p1_deep_tail01.json

search_param_match nodes=300000 games<=600 concurrency=7 openings=1200 sprt=(0.0, 3.0)
baseline={'LMR_BASE_OFFSET': 463, 'LMR_HISTORY_SCALE': 167, 'LMR_CUTNODE_BONUS': 2117, 'LMR_TTCAPTURE_BONUS': 1035, 'LMR_MOVECOUNT_FACTOR': 38, 'LMR_TABLE_SCALE_PERCENT': 100}
candidate={'LMR_BASE_OFFSET': 435, 'LMR_HISTORY_SCALE': 205, 'LMR_CUTNODE_BONUS': 1848, 'LMR_TTCAPTURE_BONUS': 1188, 'LMR_MOVECOUNT_FACTOR': 35, 'LMR_TABLE_SCALE_PERCENT': 113}
[pool] allocate worker pair 1/7 (tt=16MB)...
[pool] allocate worker pair 2/7 (tt=16MB)...
[pool] allocate worker pair 3/7 (tt=16MB)...
[pool] allocate worker pair 4/7 (tt=16MB)...
[pool] allocate worker pair 5/7 (tt=16MB)...
[pool] allocate worker pair 6/7 (tt=16MB)...
[pool] allocate worker pair 7/7 (tt=16MB)...
[warmup] JIT nodes=5000 (may take several minutes first time)...
[warmup] done in 1.3s
[warmup] JIT nodes=1500 (may take several minutes first time)...
[warmup] done in 0.0s
[warmup] JIT nodes=800 (may take several minutes first time)...
[warmup

In [ ]:
# 6. 多版本 L2 戰果橫向對照檢視（Tail 0.5 vs Tail 0.3 vs Tail 0.1）
import json
from pathlib import Path

runs = [
    ("Tail 0.5 (前次驗證)", "tournament_analysis/l2_verify_lmr_p1_deep.json"),
    ("Tail 0.3 (本次驗證)", "tournament_analysis/l2_verify_lmr_p1_deep_tail03.json"),
    ("Tail 0.1 (本次驗證)", "tournament_analysis/l2_verify_lmr_p1_deep_tail01.json"),
]

print("=" * 70)
print(f"{'版本':<22} | {'局數':<6} | {'勝率/得分率':<12} | {'Elo 表現':<18} | {'SPRT 狀態'}")
print("-" * 70)

for label, p in runs:
    path = Path(p)
    if path.exists():
        data = json.loads(path.read_text(encoding="utf-8"))
        games = data.get("games", 0)
        score = data.get("candidate_score", 0.0)
        elo = data.get("candidate_elo", 0.0)
        err = data.get("candidate_elo_err95", 0.0)
        decision = data.get("sprt", {}).get("decision", "running")
        print(f"{label:<20} | {games:<6} | {score*100:>6.2f}%      | {elo:+5.1f} +/- {err:<5.1f} | {decision}")
    else:
        print(f"{label:<20} | 尚未有資料 ({p})")
print("=" * 70)